# Query Export for Tableau Dashboard

This notebook connects to `airline_delay_analytics` and runs each analytical SQL
query stored in `sql/`, exporting the results to CSV files in `data/exports/`.

Tableau Public doesn't support live database connections (that's a Tableau Desktop
feature) — it only accepts file-based sources like CSV, Excel, or Google Sheets.
Exporting query results here keeps a single source of truth for the SQL logic (the
`.sql` files) while still producing dashboard-ready files.

**Queries exported:**
- `carrier_scorecard` — on-time %, avg delay, and cancellation rate by airline
- `delay_causes_breakdown` — root-cause share of total delay minutes
- `monthly_trends` — seasonal on-time %, delay, and cancellation patterns
- `route_reliability` — least reliable origin-destination pairs (200+ flight minimum)
- `day_of_week_trends` — on-time % and delay by day of week
- `carrier_ranking` — window function (`RANK`) example, ranking carriers by on-time %
- `month_over_month_change` — CTE + window function (`LAG`) example, tracking
  month-over-month change in on-time %

In [5]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [3]:
load_dotenv('../.env')

engine = create_engine(
    f'postgresql://{os.getenv("DB_USER")}@{os.getenv("DB_HOST")}:{os.getenv("DB_PORT")}/{os.getenv("DB_NAME")}'
)

In [9]:
queries = {
    'carrier_ranking': open('../sql/carrier_ranking.sql').read(),
    'carrier_scorecard': open('../sql/carrier_scorecard.sql').read(),
    'day_of_week_trends': open('../sql/day_of_week_trends.sql').read(),
    'delay_causes_breakdown': open('../sql/delay_causes_breakdown.sql').read(),
    'monthly_trends': open('../sql/monthly_trends.sql').read(),
    'month_over_month_change': open('../sql/month_over_month_change.sql').read(),
    'route_reliability': open('../sql/route_reliability.sql').read(),
}

for name, sql in queries.items():
    df = pd.read_sql(text(sql), engine)
    df.to_csv(f'../data/exports/{name}.csv', index=False)
    print(f'{name}: {df.shape}')

carrier_ranking: (14, 3)
carrier_scorecard: (14, 5)
day_of_week_trends: (7, 5)
delay_causes_breakdown: (1, 5)
monthly_trends: (12, 5)
month_over_month_change: (12, 4)
route_reliability: (20, 5)
